# Lab 15 · Problem framing and typed message contracts

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Frame checkout resilience before the council votes
**Start with a problem:** Reviewers may interpret a request to make checkout “fast” differently and return answers that are hard to compare.

**In this lab:** Define measurable scenarios and a common proposal shape before comparing the responses.

**New:** a chair turns a broad goal into measurable quality scenarios, and every architect replies through one proposal contract.

We revisit the checkout exercise: Inventory is usually 50 ms but can take 5 seconds; Payments has p99 1.5 seconds and the charge is not idempotent; Recommendations is optional. Measurements describe different dependencies, so do not use one timeout or retry policy for all of them. The numerical acceptance targets below are classroom assumptions to discuss, not extra claims from the example.

**At a glance**

![Lab 15 at a glance: Brief (ShopEasy checkout), then Chair frames (3 ISO 25010 scenarios), then has numbers?, then 4 architects (one contract), then Contract check (option · marks · confidence), then Aggregate (votes vs confidence).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/15.svg)

## Think about the example

Checkout depends on critical Inventory and Payments services and optional Recommendations. Their latency and failure effects differ; a payment charge is not idempotent. The example's targets are assumptions to validate.

**Concepts:** Dependency-specific resilience; Non-idempotent payment safety; Requirements versus assumed targets.

Use the scenario above to consider these questions before running the code:

1. Why can the same timeout not fit every checkout dependency?
2. Can a well-formed proposal still recommend an unsafe payment retry?

## The idea in context

Framing gives participants the same question and quality goals. A typed contract makes replies comparable but cannot establish that a proposed target is appropriate.

Timeouts, retries, breakers, bulkheads, and fallback solve different failure problems.

| Key term | Meaning |
| --- | --- |
| **Quality-attribute scenario** | A testable statement: a stimulus, the required response, and a measure. |
| **ISO/IEC 25010** | A standard model of software quality characteristics. |
| **Message contract** | The fields and rules that every message between agents must follow. |
| **Self-reported confidence** | A model's own estimate of how sure it is; information, not proof. |

## Pattern · Passive Goal Creator

Structure a broad goal into a frame before taking action. Typed proposal messages make independent reviewers comparable. The trade-off is false confidence if a shallow format check is mistaken for proof that a design meets its quality targets.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 12 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/fc798d4d9fda69756f7d860756fe7c0507f7f2a2.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER, max_tokens=768)

## Step 1 · Name qualities and check scenario structure

The quality labels are a small shared vocabulary. Each scenario needs a stimulus, response, and numerical measure. A number makes a target discussable; a schema check cannot prove the target is feasible.

The brief combines case facts and explicit classroom targets. Ask a stakeholder to confirm targets before treating them as real requirements.

In [ ]:
import json

QUALITIES = ["performance efficiency", "reliability", "maintainability"]
brief = ("ShopEasy checkout: Inventory usually takes 50 ms but spikes to 5 s. "
         "Payment p99 is 1.5 s and charging is NOT idempotent. Recommendations is optional. "
         "Classroom targets: optional failure never blocks checkout; no blind charge retry; inventory wait bounded to 1 s.")

def scenario_problems(s):
    if not isinstance(s, dict):
        return ["scenario must be an object"]
    problems = [f"missing {k}" for k in ("quality", "stimulus", "response", "measure") if not s.get(k)]
    if s.get("quality") not in QUALITIES:
        problems.append("unknown quality")
    if not any(ch.isdigit() for ch in str(s.get("measure", ""))):
        problems.append("measure needs a number")
    return problems

print(scenario_problems({"quality": "speed", "measure": "fast"}))

## Step 2 · Generate three scenarios

Keep the supplied measurements separate from proposed acceptance targets. One second is an illustrative Inventory budget, not a timeout for Payments, whose p99 is already 1.5 seconds. A production budget must be measured and agreed.

In [ ]:
prompt = (f"Brief: {brief}\nFrame 3 quality scenarios using {QUALITIES}. Use only the supplied facts and targets. "
          'Reply with a JSON list of objects containing quality, stimulus, response, measure. Each measure must contain at least one numeric digit; use 0 for zero. No code fences.')
scenarios = json.loads(ask(prompt))
assert isinstance(scenarios, list) and len(scenarios) == 3
for scenario in scenarios:
    print(scenario, scenario_problems(scenario))
assert not any(scenario_problems(s) for s in scenarios)

## Step 3 · Define a proposal contract

Every proposal names an option, a confidence, one mark per scenario, and a claim. Marks use scenario indices because two scenarios may share a quality label. A boolean is not accepted as confidence even though Python treats it as an integer.

The contract checks the reply shape, not the truth of a claim. A high confidence number does not make an unsafe option acceptable.

In [ ]:
OPTIONS = ["guarded checkout", "blind retries everywhere"]
FORMAT = ('JSON only: {"option": ..., "confidence": 0 to 1, "marks": {"0": "+ or -", "1": "+ or -", '
          '"2": "+ or -"}, "claim": "one sentence"}')

def contract_problems(msg):
    if not isinstance(msg, dict):
        return ["proposal must be an object"]
    confidence, marks = msg.get("confidence"), msg.get("marks")
    valid_marks = isinstance(marks, dict) and all(marks.get(str(i)) in ("+", "-") for i in range(3))
    rules = {"unknown option": msg.get("option") in OPTIONS,
             "mark every scenario": valid_marks,
             "confidence must be a number from 0 to 1": type(confidence) in (int, float) and 0 <= confidence <= 1,
             "claim must be nonempty text": isinstance(msg.get("claim"), str) and bool(msg["claim"].strip())}
    return [problem for problem, ok in rules.items() if not ok]

print(contract_problems({"option": "serverless", "confidence": "high"}))

## Step 4 · Four reviewers use the same contract

These names are stances, not different jobs: each reviews the complete resilience choice. All four use the selected model with different role prompts; any malformed proposal is rejected. Passing this contract does not verify the three marks; inspect each claim against the checkout exercise.

In [ ]:
architects = {"Minimalist": "Prefer small operational overhead.", "Safeguard": "Prevent duplicate charges.",
              "Modernizer": "Keep checkout responsive.", "Auditor": "Demand measured acceptance criteria."}
accepted = {}
for name, stance in architects.items():
    task = f"You are {name}. {stance}\nBrief: {brief}\nScenarios: {scenarios}\nChoose one of {OPTIONS}. {FORMAT}"
    reply = json.loads(ask(task))
    problems = contract_problems(reply)
    print(name, problems or reply["option"])
    if not problems:
        accepted[name] = reply

## Step 5 · Compare votes and confidence

Self-reported confidence is a claim, not measured trust. One confident unsafe answer must not bypass a safety rule. In a real implementation, independently reject non-idempotent blind retries before executing any payment action.

A sum of self-reported confidence is a teaching illustration, not a calibrated probability or a substitute for evidence.

In [ ]:
votes, confidence = {}, {}
for proposal in accepted.values():
    option = proposal["option"]
    votes[option] = votes.get(option, 0) + 1
    confidence[option] = round(confidence.get(option, 0) + proposal["confidence"], 2)
print("votes:", votes, "| confidence sums:", confidence)

## What you should see

The vague frame and malformed proposal fail their checks. Each model proposal is checked before votes are counted. The accepted proposals and votes can vary; passing the shape contract does not prove an architecture recommendation is safe.

## Try it

1. Explain why Inventory's one-second classroom budget cannot be copied to Payments.
2. Change a scenario's measure to `fast`. Which gate catches it?
3. Send `true` as confidence, or `[]` as the whole proposal. Both must be rejected.
4. **Use it in your council.** Frame the ShopEasy Returns and Refunds homework with measurable targets. Mark which are supplied requirements and which are assumptions needing human approval.

## Takeaway

Frame the domain decision before collecting opinions. A common message contract supports comparison and substitution; executable safety constraints and human judgement must still check the design itself.